In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: HS, HRPT

This example demonstrates a Rietveld refinement of HS crystal
structure using constant wavelength neutron powder diffraction data
from HRPT at PSI.

## 🛠️ Import Library

In [2]:
from easydiffraction import ExperimentFactory
from easydiffraction import Project
from easydiffraction import StructureFactory
from easydiffraction import download_data

## 🧩 Define Structure

This section shows how to add structures and modify their
parameters.

### Create Structure

In [3]:
structure = StructureFactory.from_scratch(name='hs')

### Set Space Group

In [4]:
structure.space_group.name_h_m = 'R -3 m'
structure.space_group.coord_system_code = 'h'

### Set Unit Cell

In [5]:
structure.cell.length_a = 6.85
structure.cell.length_c = 14.1

### Set Atom Sites

In [6]:
structure.atom_sites.create(
    id='Zn',
    type_symbol='Zn',
    fract_x=0,
    fract_y=0,
    fract_z=0.5,
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='Cu',
    type_symbol='Cu',
    fract_x=0.5,
    fract_y=0,
    fract_z=0,
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='O',
    type_symbol='O',
    fract_x=0.21,
    fract_y=-0.21,
    fract_z=0.06,
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='Cl',
    type_symbol='Cl',
    fract_x=0,
    fract_y=0,
    fract_z=0.197,
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='H',
    type_symbol='2H',
    fract_x=0.13,
    fract_y=-0.13,
    fract_z=0.08,
    adp_iso=0.5,
)

## 🔬 Define Experiment

This section shows how to add experiments, configure their parameters,
and link the structures defined in the previous step.

### Download Data

In [7]:
data_path = download_data('meas-hs-hrpt', destination='data')

Getting data...


Data 'meas-hs-hrpt': HS, HRPT (PSI)


✅ Data 'meas-hs-hrpt' downloaded to '../../../data/meas-hs-hrpt.xye'


### Create Experiment

In [8]:
expt = ExperimentFactory.from_data_path(name='hrpt', data_path=data_path)

### Set Instrument

In [9]:
expt.instrument.setup_wavelength = 1.89
expt.instrument.calib_twotheta_offset = 0.0

### Set Peak Profile

In [10]:
expt.peak.show_supported()

Peak types


,,Type,Description
1,*,pseudo-voigt,CWL pseudo-Voigt profile
2,,pseudo-voigt + berar-baldinozzi asymmetry,CWL pseudo-Voigt profile with Berar-Baldinozzi asymmetry correction.


In [11]:
expt.peak.type = 'pseudo-voigt + berar-baldinozzi asymmetry'

⚠️ Switching peak profile type adds these settings with defaults:
• asym_beba_a0=0.0
• asym_beba_a1=0.0
• asym_beba_b0=0.0
• asym_beba_b1=0.0


Peak profile type for experiment 'hrpt' changed to


pseudo-voigt + berar-baldinozzi asymmetry


In [12]:
expt.peak.broad_gauss_u = 0.1
expt.peak.broad_gauss_v = -0.2
expt.peak.broad_gauss_w = 0.2
expt.peak.broad_lorentz_y = 0

In [13]:
expt.peak.cutoff_fwhm = 8

### Set Background

In [14]:
expt.background.auto_estimate()

### Set Linked Structures

In [15]:
expt.linked_structures.create(structure_id='hs', scale=0.5)

## 📦 Define Project

The project object is used to manage the structure, experiment, and
analysis.

### Create Project

In [16]:
project = Project(name='hs_hrpt')

### Add Structure

In [17]:
project.structures.add(structure)

### Add Experiment

In [18]:
project.experiments.add(expt)

## 🚀 Perform Analysis

This section shows the analysis process, including how to set up
calculation and fitting engines.


### Display Structure

In [19]:
project.display.structure(struct_name='hs')

Structure 🧩 'hs' (Atom view type: 'covalent')


### Display Pattern

In [20]:
project.display.pattern(expt_name='hrpt')

In [21]:
project.display.pattern(expt_name='hrpt', x_min=48, x_max=51)

### Perform Fit 1/4

Set parameters to be refined.

In [22]:
structure.cell.length_a.free = True
structure.cell.length_c.free = True

expt.linked_structures['hs'].scale.free = True
expt.instrument.calib_twotheta_offset.free = True

Show free parameters after selection.

In [23]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,hs,cell,,length_a,6.85000,,-inf,inf,Å
2,hs,cell,,length_c,14.10000,,-inf,inf,Å
3,hrpt,linked_structure,hs,scale,0.50000,,-inf,inf,
4,hrpt,instrument,,twotheta_offset,0.00000,,-inf,inf,deg


#### Run Fitting

In [24]:
project.analysis.minimizer.chi_square_change_tolerance = 1e-2

In [25]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.05,210.23,
2,8,0.39,60.12,71.4% ↓
3,13,0.95,55.45,7.8% ↓
4,18,1.28,51.96,6.3% ↓
5,23,1.52,49.78,4.2% ↓
6,28,1.93,48.53,2.5% ↓
7,33,2.51,47.85,1.4% ↓
8,39,2.82,47.49,


🏆 Best goodness-of-fit (reduced χ²) is 47.49 at iteration 38


✅ Fitting complete.


In [26]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.
2,chi_square_change_tolerance,0.01,Relative change in the objective (chi-square) used to stop fitting.
3,parameter_change_tolerance,1e-08,Relative change in fitted parameters used to stop fitting.
4,gradient_tolerance,0.0,Gradient orthogonality used to stop fitting; zero disables it.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),2.82
4,🔁 Iterations,36
5,📏 Goodness-of-fit (reduced χ²),47.49
6,"📏 R-factor (Rf, %)",17.81
7,"📏 R-factor squared (Rf², %)",30.41
8,"📏 Weighted R-factor (wR, %)",25.01


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,hs,cell,,length_a,Å,6.8500,6.8604,0.0003,0.15 % ↑
2,hs,cell,,length_c,Å,14.1000,14.1312,0.0008,0.22 % ↑
3,hrpt,linked_structure,hs,scale,,0.5000,0.2432,0.0029,51.36 % ↓
4,hrpt,instrument,,twotheta_offset,deg,0.0000,0.1092,0.0052,N/A


#### Display Pattern

In [27]:
project.display.pattern(expt_name='hrpt')

In [28]:
project.display.pattern(expt_name='hrpt', x_min=48, x_max=51)

### Perform Fit 2/4

Set more parameters to be refined.

In [29]:
expt.peak.broad_gauss_u.free = True
expt.peak.broad_gauss_v.free = True
expt.peak.broad_gauss_w.free = True
expt.peak.broad_lorentz_y.free = True

for point in expt.background:
    point.intensity.free = True

Show free parameters after selection.

In [30]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,hs,cell,,length_a,6.86041,0.00030,-inf,inf,Å
2,hs,cell,,length_c,14.13115,0.00085,-inf,inf,Å
3,hrpt,linked_structure,hs,scale,0.24321,0.00292,-inf,inf,
4,hrpt,peak,,broad_gauss_u,0.10000,,-inf,inf,deg²
5,hrpt,peak,,broad_gauss_v,-0.20000,,-inf,inf,deg²
6,hrpt,peak,,broad_gauss_w,0.20000,,-inf,inf,deg²
7,hrpt,peak,,broad_lorentz_y,0.00000,,-inf,inf,deg
8,hrpt,instrument,,twotheta_offset,0.10920,0.00519,-inf,inf,deg
9,hrpt,background,1,intensity,645.00000,,-inf,inf,
10,hrpt,background,2,intensity,460.00000,,-inf,inf,


#### Run Fitting

In [31]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.04,47.73,
2,24,1.34,16.34,65.8% ↓
3,45,2.81,15.68,4.0% ↓
4,66,4.29,13.04,16.8% ↓
5,88,5.82,12.92,


🏆 Best goodness-of-fit (reduced χ²) is 12.92 at iteration 87


✅ Fitting complete.


In [32]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.
2,chi_square_change_tolerance,0.01,Relative change in the objective (chi-square) used to stop fitting.
3,parameter_change_tolerance,1e-08,Relative change in fitted parameters used to stop fitting.
4,gradient_tolerance,0.0,Gradient orthogonality used to stop fitting; zero disables it.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),5.82
4,🔁 Iterations,85
5,📏 Goodness-of-fit (reduced χ²),12.92
6,"📏 R-factor (Rf, %)",9.89
7,"📏 R-factor squared (Rf², %)",13.84
8,"📏 Weighted R-factor (wR, %)",13.01


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,hs,cell,,length_a,Å,6.8604,6.8626,0.0003,0.03 % ↑
2,hs,cell,,length_c,Å,14.1312,14.1392,0.0009,0.06 % ↑
3,hrpt,linked_structure,hs,scale,,0.2432,0.4173,0.0039,71.58 % ↑
4,hrpt,peak,,broad_gauss_u,deg²,0.1000,0.3637,0.0271,263.67 % ↑
5,hrpt,peak,,broad_gauss_v,deg²,-0.2000,-0.2912,0.0483,45.62 % ↑
6,hrpt,peak,,broad_gauss_w,deg²,0.2000,0.2240,0.0204,11.98 % ↑
7,hrpt,peak,,broad_lorentz_y,deg,0.0000,0.1621,0.0099,N/A
8,hrpt,instrument,,twotheta_offset,deg,0.1092,0.1282,0.0039,17.40 % ↑
9,hrpt,background,1,intensity,,645.0000,673.1456,20.0373,4.36 % ↑
10,hrpt,background,2,intensity,,460.0000,454.8721,6.2706,1.11 % ↓


#### Display Pattern

In [33]:
project.display.pattern(expt_name='hrpt')

In [34]:
project.display.pattern(expt_name='hrpt', x_min=48, x_max=51)

### Perform Fit 3/4

Set more parameters to be refined.

In [35]:
structure.atom_sites['O'].fract_x.free = True
structure.atom_sites['O'].fract_z.free = True
structure.atom_sites['Cl'].fract_z.free = True
structure.atom_sites['H'].fract_x.free = True
structure.atom_sites['H'].fract_z.free = True

Show free parameters after selection.

In [36]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,hs,cell,,length_a,6.86265,0.00034,-inf,inf,Å
2,hs,cell,,length_c,14.13920,0.00092,-inf,inf,Å
3,hs,atom_site,O,fract_x,0.21000,,-inf,inf,
4,hs,atom_site,O,fract_z,0.06000,,-inf,inf,
5,hs,atom_site,Cl,fract_z,0.19700,,-inf,inf,
6,hs,atom_site,H,fract_x,0.13000,,-inf,inf,
7,hs,atom_site,H,fract_z,0.08000,,-inf,inf,
8,hrpt,linked_structure,hs,scale,0.41730,0.00391,-inf,inf,
9,hrpt,peak,,broad_gauss_u,0.36367,0.02709,-inf,inf,deg²
10,hrpt,peak,,broad_gauss_v,-0.29125,0.04827,-inf,inf,deg²


#### Run Fitting

In [37]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.05,12.94,
2,29,1.77,5.19,59.9% ↓
3,55,3.49,4.95,4.6% ↓
4,82,5.85,4.95,


🏆 Best goodness-of-fit (reduced χ²) is 4.95 at iteration 81


✅ Fitting complete.


In [38]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.
2,chi_square_change_tolerance,0.01,Relative change in the objective (chi-square) used to stop fitting.
3,parameter_change_tolerance,1e-08,Relative change in fitted parameters used to stop fitting.
4,gradient_tolerance,0.0,Gradient orthogonality used to stop fitting; zero disables it.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),5.85
4,🔁 Iterations,79
5,📏 Goodness-of-fit (reduced χ²),4.95
6,"📏 R-factor (Rf, %)",6.42
7,"📏 R-factor squared (Rf², %)",8.75
8,"📏 Weighted R-factor (wR, %)",8.05


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,hs,cell,,length_a,Å,6.8626,6.8620,0.0002,0.01 % ↓
2,hs,cell,,length_c,Å,14.1392,14.1349,0.0005,0.03 % ↓
3,hs,atom_site,O,fract_x,,0.2100,0.2059,0.0002,1.95 % ↓
4,hs,atom_site,O,fract_z,,0.0600,0.0629,0.0002,4.80 % ↑
5,hs,atom_site,Cl,fract_z,,0.1970,0.1974,0.0002,0.20 % ↑
6,hs,atom_site,H,fract_x,,0.1300,0.1339,0.0002,2.99 % ↑
7,hs,atom_site,H,fract_z,,0.0800,0.0871,0.0001,8.86 % ↑
8,hrpt,linked_structure,hs,scale,,0.4173,0.3950,0.0023,5.34 % ↓
9,hrpt,peak,,broad_gauss_u,deg²,0.3637,0.3600,0.0143,1.01 % ↓
10,hrpt,peak,,broad_gauss_v,deg²,-0.2912,-0.3846,0.0271,32.06 % ↑


#### Display Pattern

In [39]:
project.display.pattern(expt_name='hrpt')

In [40]:
project.display.pattern(expt_name='hrpt', x_min=48, x_max=51)

### Perform Fit 4/4

Set more parameters to be refined.

In [41]:
structure.atom_sites['Zn'].adp_iso.free = True
structure.atom_sites['Cu'].adp_iso.free = True
structure.atom_sites['O'].adp_iso.free = True
structure.atom_sites['Cl'].adp_iso.free = True
structure.atom_sites['H'].adp_iso.free = True

expt.peak.asym_beba_a0.free = True
expt.peak.asym_beba_b0.free = True
expt.peak.asym_beba_a1.free = True
expt.peak.asym_beba_b1.free = True

Show free parameters after selection.

In [42]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,hs,cell,,length_a,6.86205,0.00022,-inf,inf,Å
2,hs,cell,,length_c,14.13486,0.00053,-inf,inf,Å
3,hs,atom_site,Zn,adp_iso,0.50000,,-inf,inf,Å²
4,hs,atom_site,Cu,adp_iso,0.50000,,-inf,inf,Å²
5,hs,atom_site,O,fract_x,0.20591,0.00024,-inf,inf,
6,hs,atom_site,O,fract_z,0.06288,0.00018,-inf,inf,
7,hs,atom_site,O,adp_iso,0.50000,,-inf,inf,Å²
8,hs,atom_site,Cl,fract_z,0.19740,0.00018,-inf,inf,
9,hs,atom_site,Cl,adp_iso,0.50000,,-inf,inf,Å²
10,hs,atom_site,H,fract_x,0.13389,0.00018,-inf,inf,


#### Run Fitting

In [43]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.06,4.96,
2,38,3.04,2.15,56.6% ↓
3,73,5.27,1.95,9.2% ↓
4,109,8.98,1.95,


🏆 Best goodness-of-fit (reduced χ²) is 1.95 at iteration 108


✅ Fitting complete.


In [44]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.
2,chi_square_change_tolerance,0.01,Relative change in the objective (chi-square) used to stop fitting.
3,parameter_change_tolerance,1e-08,Relative change in fitted parameters used to stop fitting.
4,gradient_tolerance,0.0,Gradient orthogonality used to stop fitting; zero disables it.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),8.98
4,🔁 Iterations,106
5,📏 Goodness-of-fit (reduced χ²),1.95
6,"📏 R-factor (Rf, %)",3.99
7,"📏 R-factor squared (Rf², %)",4.64
8,"📏 Weighted R-factor (wR, %)",5.05


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,hs,cell,,length_a,Å,6.8620,6.8639,0.0004,0.03 % ↑
2,hs,cell,,length_c,Å,14.1349,14.1417,0.0009,0.05 % ↑
3,hs,atom_site,Zn,adp_iso,Å²,0.5000,0.1678,0.0604,66.43 % ↓
4,hs,atom_site,Cu,adp_iso,Å²,0.5000,1.4870,0.0380,197.40 % ↑
5,hs,atom_site,O,fract_x,,0.2059,0.2062,0.0002,0.17 % ↑
6,hs,atom_site,O,fract_z,,0.0629,0.0611,0.0001,2.91 % ↓
7,hs,atom_site,O,adp_iso,Å²,0.5000,1.0805,0.0374,116.10 % ↑
8,hs,atom_site,Cl,fract_z,,0.1974,0.1967,0.0001,0.35 % ↓
9,hs,atom_site,Cl,adp_iso,Å²,0.5000,1.4335,0.0380,186.69 % ↑
10,hs,atom_site,H,fract_x,,0.1339,0.1324,0.0002,1.14 % ↓


In [45]:
project.display.fit.correlations()

#### Display Pattern

In [46]:
project.display.pattern(expt_name='hrpt')

In [47]:
project.display.pattern(expt_name='hrpt', x_min=48, x_max=51)

## 📊 Report

The HTML report is written automatically when the project is saved;
enable `project.report.pdf` as well for a PDF version.

## 💾 Save Project

In [48]:
project.save_as(dir_path='projects/refine-hs-hrpt')

Saving project 📦 'hs_hrpt' to '../../../projects/refine-hs-hrpt'


├── 📄 project.edi
├── 📁 structures/
│   └── 📄 hs.edi
├── 📁 experiments/
│   └── 📄 hrpt.edi
├── 📁 analysis/
│   └── 📄 analysis.edi
└── 📁 reports/
    └── 📄 hs_hrpt.html
